<a href="https://colab.research.google.com/github/gustavo159753/Analise-de-dados/blob/main/tempUmiAnalisys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Importando as bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

#Importando o arquivo.csv

In [ ]:
upload = files.upload()
files.view
print(f'Arquivo {upload} carregado com sucesso.')

#Convertendo em df e realizando tratamentos

In [ ]:
import io

# Obtendo o nome do primeiro arquivo carregado no dicionário de upload
file_name = list(upload.keys())[0]

# Lendo o conteúdo do arquivo (em memória) e convertendo para um DataFrame do Pandas
# Usamos decimal=',' pois os dados numéricos utilizam a vírgula como separador decimal
df = pd.read_csv(io.StringIO(upload[file_name].decode('utf-8')), decimal=',')

# Convertendo a coluna de data para o formato datetime, ignorando erros de texto como 'Coluna 1'
df['Timestamp'] = pd.to_datetime(df['Temperatura (ºC)'], dayfirst=True, errors='coerce')

# Removendo linhas inválidas e criando uma cópia limpa do DataFrame
df = df.dropna(subset=['Timestamp']).copy()

# Extraindo Hora, Minuto e Data para as análises temporais posteriores
df['Hour'] = df['Timestamp'].dt.hour
df['Minute'] = df['Timestamp'].dt.minute
df['Date'] = df['Timestamp'].dt.date

In [ ]:
df.head()

#Primeira leitura do gráfico antes da personalização

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='T', y='U', label='Dados Medidos', alpha=0.6)
sns.scatterplot(data=df, x='T padrão', y='U padrão', label='Dados Padrão', marker='X', s=100, color='red')
plt.axvline(x=25, color='green', linestyle='--', label='Temperatura Máxima (25°C)')
plt.axhline(y=58, color='red', linestyle='--', label='Umidade máxima (58%)')
plt.title('Relação Umidade (%) vs Temperatura (°C)')
plt.xlabel('Temperatura (°C)')
plt.ylabel('Umidade (%)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.show()

#Tratamento da linha de leitura padrão de T e U

In [ ]:
# Criando um DataFrame auxiliar para ordenar os dados padrão por temperatura
df_padrao = df[['T padrão', 'U padrão']].dropna().sort_values('T padrão')

plt.figure(figsize=(12, 7))

# Dados medidos
sns.scatterplot(data=df, x='T', y='U', label='Dados Medidos', alpha=0.4, color='steelblue')

# Linha Padrão (Referência)
sns.lineplot(data=df_padrao, x='T padrão', y='U padrão', label='Linha Padrão (Referência)', color='red', linewidth=2, marker='X')

# Limites Críticos
plt.axvline(x=25, color='green', linestyle='--', label='Temp. Máxima (25°C)')
plt.axhline(y=58, color='darkred', linestyle='--', label='Umid. Máxima (58%)')

plt.title('Análise de Desvio: Medição vs Padrão')
plt.xlabel('Temperatura (°C)')
plt.ylabel('Umidade (%)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

#Tabela de estudo de pontos de T e U para análise

In [ ]:
# Agrupando os dados pelo Status para contar a frequência de cada categoria
contagem_status = df['Status'].value_counts().reset_index()
contagem_status.columns = ['Classificação', 'Quantidade de Pontos']

# Calculando a porcentagem de cada categoria sobre o volume total de dados
total_pontos = contagem_status['Quantidade de Pontos'].sum()
contagem_status['Porcentagem (%)'] = (contagem_status['Quantidade de Pontos'] / total_pontos * 100).round(2)

# Exibindo a tabela resumo final para o relatório
display(contagem_status)

#Realizando o Storitelling dos dados em um gráfico personalizado

In [ ]:
import numpy as np

# 1. Preparando a tabela de referência (padrão) removendo valores vazios
ref = df[['T padrão', 'U padrão']].dropna().sort_values('T padrão')

# 2. Definindo a lógica de classificação para cada ponto medido
def classificar(row):
    t_val = row['T']
    u_val = row['U']

    # Interpolação: calcula qual deveria ser a umidade padrão para a temperatura atual do registro
    u_esperada = np.interp(t_val, ref['T padrão'], ref['U padrão'])

    # Verificando se os valores ultrapassam os limites técnicos
    t_fora = t_val > 25
    u_fora = u_val > u_esperada

    # Classificação baseada nos desvios encontrados
    if not t_fora and not u_fora:
        return 'Dentro do Padrão'
    elif t_fora and not u_fora:
        return 'T fora do Padrão (>25°C)'
    elif not t_fora and u_fora:
        return 'U fora do Padrão (Acima da Ref)'
    else:
        return 'T e U fora do Padrão'

# Criando a nova coluna 'Status' no DataFrame original
df['Status'] = df.apply(classificar, axis=1)

# 3. Configuração visual do gráfico ampliado
plt.figure(figsize=(18, 10))

# Definindo as cores para cada categoria de conformidade
palette = {
    'Dentro do Padrão': 'green',
    'U fora do Padrão (Acima da Ref)': 'orange',
    'T fora do Padrão (>25°C)': 'magenta',
    'T e U fora do Padrão': 'red'
}

# Plotando os pontos medidos coloridos pelo Status
sns.scatterplot(data=df, x='T', y='U', hue='Status', palette=palette, alpha=0.7, s=60)

# Adicionando a linha de referência preta (o 'ideal' a ser seguido)
sns.lineplot(data=ref, x='T padrão', y='U padrão', color='black', label='Linha de Referência (Padrão)', linewidth=3, linestyle='--')

# Desenhando as linhas de limites críticos (25°C e 58% de Umidade)
plt.axvline(x=25, color='gray', linestyle=':', label='Limite Crítico 25°C', linewidth=2)
plt.axhline(y=58, color='darkred', linestyle='--', label='Umid. Máxima (58%)')

# Ajustes de títulos, eixos e legendas para facilitar a leitura dos diretores
plt.title('Classificação de Conformidade dos Dados (T vs U)', fontsize=20, pad=20)
plt.xlabel('Temperatura (°C)', fontsize=16)
plt.ylabel('Umidade (%)', fontsize=16)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0, fontsize=14, markerscale=3)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#extração das datas e a identificação dos dias únicos no conjunto de dados.

In [ ]:
# 1. Extrair a parte da data da coluna Timestamp
df['Date'] = df['Timestamp'].dt.date

# 2. Criar uma lista ordenada de dias únicos
unique_days = sorted(df['Date'].unique())

# 3. Imprimir os dias únicos para verificar a cobertura dos dados
print(f'Dias únicos encontrados no conjunto de dados: {unique_days}')
display(df[['Timestamp', 'Date']].head())

#Leitura de cada ponto por dia no gráfico Hora X minuto (distribuição temporal)

In [ ]:
for day in unique_days:
    # Filtra os dados apenas para o dia atual do loop
    df_day = df[df['Date'] == day]

    # Define o tamanho da figura para o gráfico do dia
    plt.figure(figsize=(15, 6))

    # Cria o gráfico de dispersão: Minutos no eixo X, Horas no eixo Y e cores baseadas no Status
    sns.scatterplot(data=df_day, x='Minute', y='Hour', hue='Status', palette=palette, alpha=0.7, s=80)

    # Define um título dinâmico contendo a data atual
    plt.title(f'Distribuição Temporal de Status - {day}', fontsize=16, pad=15)

    # Configura os nomes dos eixos
    plt.xlabel('Minuto', fontsize=12)
    plt.ylabel('Hora', fontsize=12)

    # Define as marcações dos eixos para mostrar todas as 24 horas e intervalos de 5 minutos
    plt.yticks(range(0, 24))
    plt.xticks(range(0, 61, 5))

    # Ajusta os limites dos eixos para melhor visualização
    plt.ylim(-0.5, 23.5)
    plt.xlim(-1, 60)

    # Posiciona a legenda fora do gráfico para não obstruir os dados
    plt.legend(title='Status', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)

    # Adiciona uma grade suave para facilitar a leitura dos pontos
    plt.grid(True, linestyle='--', alpha=0.4)

    # Ajusta o layout para evitar cortes nos textos e exibe o gráfico
    plt.tight_layout()
    plt.show()